# Homework 2 - Training and Optimization

In [ ]:
import tensorflow as tf
from reader import AudioReader
from preprocessing import Padding, Normalization
from preprocessing import MelSpectrogram

### Define the hyperparameters

In [ ]:
PREPROCESSING_ARGS = {
    'sampling_rate': 16000,
    'frame_length_in_s': 0.008,
    'frame_step_in_s': 0.002,
    'num_mel_bins': 40,
    'lower_frequency': 20,
    'upper_frequency': 4000,
}

TRAINING_ARGS = {
    'batch_size': 20,
    'learning_rate': 0.01,
    'end_learning_rate': 1.e-5,
    'epochs': 20
}

### Create train/val/test Datasets and Callback

In [ ]:
train_ds = tf.data.Dataset.list_files(['/tmp/msc-train/down*', '/tmp/msc-train/up*'])
val_ds = tf.data.Dataset.list_files(['/tmp/msc-val/down*', '/tmp/msc-val/up*'])
test_ds = tf.data.Dataset.list_files(['/tmp/msc-test/down*', '/tmp/msc-test/up*'])

In [ ]:
# Learning Rate scheduler
linear_decay = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=TRAINING_ARGS['learning_rate'],
    end_learning_rate=TRAINING_ARGS['end_learning_rate'],
    decay_steps=len(train_ds) * TRAINING_ARGS['epochs'],
)
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(linear_decay)

# Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    verbose=1,
    mode='auto'
)

### Define the data pipeline: Log-Mel

In [ ]:
audio_reader = AudioReader(tf.int16)
padding = Padding(PREPROCESSING_ARGS['sampling_rate'])
normalization = Normalization(tf.int16)
mel_spec_processor = MelSpectrogram(**PREPROCESSING_ARGS)

LABELS = ['down', 'up']

def prepare_for_training(feature, label):
    feature = tf.expand_dims(feature, -1)
    label_id = tf.argmax(label == LABELS)

    return feature, label_id

train_ds = (train_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(mel_spec_processor.get_mel_spec_and_label)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size'])
            .cache())
val_ds = (val_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(mel_spec_processor.get_mel_spec_and_label)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size']))
test_ds = (test_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(mel_spec_processor.get_mel_spec_and_label)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size']))

### Define the data pipeline 

### Read a batch of data

In [ ]:
for example_batch, example_labels in train_ds.take(1):
  print('Batch Shape:', example_batch.shape)
  print('Data Shape:', example_batch.shape[1:])
  print('Labels:', example_labels)
     

### Create the model: Log-Mel

In [ ]:
model_LM = tf.keras.Sequential([
    tf.keras.layers.Input(shape=example_batch.shape[1:]),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[2, 2], use_bias=False, padding='valid'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[1, 1], use_bias=False, padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[1, 1], use_bias=False, padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(units=len(LABELS)),
    tf.keras.layers.Softmax()
])

In [ ]:
model.summary()

### Train the model with callback

In [ ]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=False)
optimizer = tf.optimizers.Adam(learning_rate=linear_decay)
metrics = [tf.metrics.SparseCategoricalAccuracy()]
model.compile(loss=loss, optimizer=optimizer, metrics=metrics)

history = model.fit(
    train_ds, 
    epochs=TRAINING_ARGS['epochs'], 
    validation_data=val_ds, 
    callbacks=[lr_scheduler, early_stopping]
)

### Show the history

In [ ]:
history.history

### Evaluate the model

In [ ]:
training_loss = history.history['loss'][-1]
training_accuracy = history.history['sparse_categorical_accuracy'][-1]
val_loss = history.history['val_loss'][-1]
val_accuracy = history.history['val_sparse_categorical_accuracy'][-1]

test_loss, test_accuracy = model.evaluate(test_ds)

print(f'Training Loss: {training_loss:.4f}')
print(f'Training Accuracy: {training_accuracy*100.:.2f}%')
print()
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy*100.:.2f}%')
print()
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy*100.:.2f}%')

### Save hyperparameters & results

In [ ]:
import pandas as pd

output_dict = {
    'timestamp': timestamp,
    **PREPROCESSING_ARGS,
    **TRAINING_ARGS,
    'test_accuracy': test_accuracy
}

df = pd.DataFrame([output_dict])

output_path='./mel_spectrogram_results.csv'
df.to_csv(output_path, mode='a', header=not os.path.exists(output_path), index=False)

### TFLite Conversion

In [ ]:
# Converting a SavedModel to a TensorFlow Lite model.
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
tflite_model = converter.convert()

tflite_model_dir = './tflite_models'
# create the path if it doesn't exist
if not os.path.exists(tflite_model_dir):
    os.makedirs(tflite_model_dir)

# name the model
tflite_model_name = os.path.join(tflite_model_dir, f'{timestamp}.tflite')
tflite_model_name

# write the model
with open(tflite_model_name, 'wb') as fp:
    fp.write(tflite_model)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b4ef5aa4-3f71-4837-91f1-c6fd9810a7ea' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>